# Week 2, day 5 (morning) — Extra practice 13 SOLUTIONS: recursion   (L06)

Every cell below was executed on the same Python the lab ships (3.13), and the
quoted output is what it actually printed.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Extra practice 13 — Recursion. Run this once.
config = {
    "db": {"host": "localhost", "port": 5432, "opts": {"ssl": True, "pool": 5}},
    "app": {"name": "lab", "debug": False},
    "version": 3,
}

sorted_ids = [2, 5, 9, 14, 21, 30, 44, 51, 68, 77]

print("sorted_ids:", sorted_ids)

### Question 1

Summing digits. -> `7`, `6`, `35`.

Base case: a single digit is its own digit sum. Recursive case: peel the
last digit off with `n % 10` and recurse on everything else, `n // 10`.
Each call makes the number strictly shorter, which is why it terminates.

`//` is doing the work. `/` would give `12.3` and the chain would never
reach a clean single digit — it would drift through floats and either
recurse forever or stop somewhere arbitrary.

This is genuinely recursive in shape but it is also a three-line loop.
Q4–Q6 are where the recursion earns its keep.

In [ ]:
def digit_sum(n):
    if n < 10:
        return n
    return n % 10 + digit_sum(n // 10)

print(digit_sum(7))
print(digit_sum(123))
print(digit_sum(98765))

### Question 2

Reversing a string. -> `''`, `a`, `noisrucer`, `True`.

The empty string is the base case, and `text[:-1]` is guaranteed to get
there because it is always one character shorter.

It agrees with `"recursion"[::-1]`, which is how you would actually write
it: one slice, no recursion, no stack. The recursive version costs one
frame per character and copies the remaining string on every call — so a
2,000-character string would raise `RecursionError`, and a 500-character
one would copy about 125,000 characters to do a job the slice does in one
pass.

Worth writing once to see the shape. Never worth shipping.

In [ ]:
def reverse(text):
    if text == "":
        return ""
    return text[-1] + reverse(text[:-1])

print(repr(reverse("")))
print(reverse("a"))
print(reverse("recursion"))
print(reverse("recursion") == "recursion"[::-1])

### Question 3

Exponentiation. -> `2 0 1 True`, `2 10 1024 True`, `3 4 81 True`.

`exp == 0` returning `1` is not a special case bolted on — it is the
foundation the whole chain multiplies up from. Get it wrong (return 0) and
every answer becomes 0.

That is the same structural point as worksheet 13 Q8, where slide 87's
factorial returns 0 for `0!` because its base case never fires. **The base
case is not the edge case; it is the answer everything else is built on.**

A negative `exp` never reaches 0 and would run away. The docstring, or an
explicit rejection, is the only guard.

In [ ]:
def power(base, exp):
    if exp == 0:
        return 1
    return base * power(base, exp - 1)

for b, e in [(2, 0), (2, 10), (3, 4)]:
    print(b, e, power(b, e), power(b, e) == b ** e)

PART B — Where recursion earns it

### Question 4

Measuring nesting depth. -> `3`, `0`, `1`.

`config` is three levels deep: the top, `db`, and `db.opts`. An empty dict
is 0 and a flat one is 1.

This cannot be written with a fixed number of loops, because the answer is
the thing you are trying to find out. That is the signature of a genuinely
recursive problem — the same reason worksheet 13 Q9's `flatten` needed
recursion and worksheet 03 Q9's two-level version did not.

`max(depth(v) for v in value.values())` recurses into every value and keeps
the deepest. The `not value` in the base case is what stops `max()` being
handed an empty sequence, which would raise `ValueError`.

In [ ]:
def depth(value):
    if not isinstance(value, dict) or not value:
        return 0
    return 1 + max(depth(v) for v in value.values())

print(depth(config))
print(depth({}))
print(depth({"a": 1}))

### Question 5

Flattening a config. -> `db.host = localhost`, `db.port = 5432`, `db.opts.ssl = True`, `db.opts.pool = 5`, `app.name = lab`, `app.debug = False`, `version = 3`, then `7 leaves`.

Dotted paths from a tree of unknown shape, which is what every config
loader, JSON flattener and settings library does.

The recursion carries **state down** through `prefix` — each level adds its
key to the path before recursing — and passes **results up** through the
returned dict. That is the general pattern for tree work.

`isinstance(item, dict)` is the branch: a dict means go deeper, anything
else is a leaf. It would not handle a list of dicts, which is where a real
implementation grows a second branch.

Seven leaves from three top-level keys. Note the depth is 3 and the *size*
is 7 — recursion depth follows the shape, not the volume, which is why this
is safe where extra practice 13 Q2 was not.

In [ ]:
def flatten_config(value, prefix=""):
    flat = {}
    for key, item in value.items():
        path = prefix + key if prefix == "" else prefix + "." + key
        if isinstance(item, dict):
            flat.update(flatten_config(item, path))
        else:
            flat[path] = item
    return flat

result = flatten_config(config)
for path, item in result.items():
    print(path, "=", item)
print(len(result), "leaves")

### Question 6

Recursive binary search. -> `21 -> 4`, `2 -> 0`, `77 -> 9`, `22 -> -1`.

Each call halves the range: compare the middle, then recurse into the left
or the right half. Ten items need at most four comparisons.

Two base cases, and both are required. `low > high` means the range is
empty and the target is not there — that is the `-1`. `values[mid] ==
target` is success. Without the first, a missing value recurses forever.

`high=None` defaulting to `len(values) - 1` keeps the first call simple
while letting the recursive calls pass explicit bounds. It is a mutable-ish
default done correctly — `None` in the signature, the real value computed
inside — for exactly the reason worksheet 10 Q4 gave.

**This only works on sorted input**, and nothing checks that. On unsorted
data it returns `-1` for values that are present.

In [ ]:
def find(values, target, low=0, high=None):
    if high is None:
        high = len(values) - 1
    if low > high:
        return -1
    mid = (low + high) // 2
    if values[mid] == target:
        return mid
    if values[mid] < target:
        return find(values, target, mid + 1, high)
    return find(values, target, low, mid - 1)

for target in [21, 2, 77, 22]:
    print(target, "->", find(sorted_ids, target))

### Question 7

Counting the work. -> `21: binary 1 steps, scan 5 steps, index 4`; `2: binary 3 steps, scan 1 steps, index 0`; `77: binary 4 steps, scan 10 steps, index 9`; `22: binary 3 steps, scan 10 steps, index -1`.

The interesting rows are the last two. For `77` — the last element — the
scan needs all ten and the binary search needs four. For `22`, which is not
there at all, the scan must check **every** item to prove it, while the
binary search rules out half the list at each step and finishes in three.

The first two rows show it is not a free win on small data: `21` happened
to be the midpoint, and `2` is the first element, which a scan finds
immediately.

The cost grows differently: doubling the list adds **one** step to the
binary search and doubles the scan. Ten items, four steps; a thousand,
ten; a million, twenty. That is the same exponential arithmetic as
worksheet 13 Q6, running in the useful direction.

In [ ]:
def find_counted(values, target, low=0, high=None, steps=0):
    if high is None:
        high = len(values) - 1
    if low > high:
        return -1, steps
    steps = steps + 1
    mid = (low + high) // 2
    if values[mid] == target:
        return mid, steps
    if values[mid] < target:
        return find_counted(values, target, mid + 1, high, steps)
    return find_counted(values, target, low, mid - 1, steps)

for target in [21, 2, 77, 22]:
    index, steps = find_counted(sorted_ids, target)
    scan = 0
    for v in sorted_ids:
        scan = scan + 1
        if v == target:
            break
    print(f"{target}: binary {steps} steps, scan {scan} steps, index {index}")

### Question 8

Euclid's algorithm. -> `48 18 -> 6`, `270 192 -> 6`, `17 5 -> 1`, `5 0 -> 5`.

Three lines, and it is one of the oldest algorithms there is. `b == 0`
is the base case; each step replaces `(a, b)` with `(b, a % b)`, and the
remainder shrinks every time, so it always terminates — quickly.

`gcd(17, 5)` is 1: they share no factor. `gcd(5, 0)` is 5, straight from
the base case, which is the mathematically correct answer and worth
checking rather than assuming.

This is the rare case where the recursive form is both the fastest and the
clearest. The iterative version is a `while b:` loop with a swap, and it is
not obviously better.

In [ ]:
def gcd(a, b):
    if b == 0:
        return a
    return gcd(b, a % b)

for a, b in [(48, 18), (270, 192), (17, 5), (5, 0)]:
    print(a, b, "->", gcd(a, b))

### Question 9

Two broken base cases. -> `35`, then `-123` from `digit_sum(-123)`, then `RecursionError: maximum recursion depth exceeded`.

Both versions are wrong for negatives, in opposite and instructive ways.

`digit_sum(-123)` **returns immediately** with `-123`, because `-123 < 10`
is true. No error, a plausible-looking number, and completely wrong — the
quiet failure this whole course keeps coming back to.

`digit_sum_zero_base(-123)` never terminates. Python's `//` rounds towards
negative infinity, so the chain runs -123, -13, -2, -1, -1, -1… and `-1 //
10` is `-1` forever. The base case `n == 0` is never reached.

So "add a base case" is not enough. **The base case must be reachable from
every input the function accepts**, and neither of these is. The fix is to
narrow what it accepts: `n = abs(n)` at the top, or a docstring saying
non-negative integers only and a caller who honours it.

In [ ]:
print(digit_sum(98765))
print(digit_sum(-123))        # returns -123 -- wrong, but it does return

def digit_sum_zero_base(n):
    if n == 0:
        return 0
    return n % 10 + digit_sum_zero_base(n // 10)

# This is SUPPOSED to raise: RecursionError: maximum recursion depth
# exceeded.
#
# Python's // rounds towards NEGATIVE INFINITY, so -123 // 10 is -13, then
# -2, then -1 -- and -1 // 10 is -1, forever. The chain never reaches 0.
#
# Both versions are broken for negatives, in opposite ways: `n < 10` returns
# nonsense silently, `n == 0` never terminates. The fix is to reject the
# input -- `n = abs(n)` at the top, or a docstring saying non-negative only.
print(digit_sum_zero_base(-123))